# Sample Data Preview — voyage.dbt_dev

Displays a 5-row sample from each dimension and fact table in the `voyage.dbt_dev` schema.

---

## Table of Contents

1. **Sample Data Preview** — 5-row samples and column listings for all `voyage.dbt_dev` tables
2. **Source Date Range Validation** — date range checks for booking, marketing, and website fact tables
3. **Summary & Findings** — consolidated date range results and observations
4. **dbt Model Generation** — staging and marts model file generation code (commented out)
5. **Website Silver vs. dbt Reconciliation** — row-count and metric reconciliation, orphan campaign check
6. **Marketing Data Mart Validation** — preview, reconciliation, domain validation, and channel performance for `fct_campaign_daily`
7. **Final Pipeline Validation** — row-count checks across all `voyage.dbt_dev` fact tables

In [0]:
tables = [
    "dim_date",
    "dim_tenant",
    "dim_campaign",
    "dim_property",
    "dim_customer",
    "fct_campaign_daily",
    "fct_marketing_funnel_daily",
    "fct_marketing_attribution_daily",
    "fct_website_daily",
    "fct_booking_daily",
    "fct_property_daily",
    "fct_tenant_performance",
]

for tbl in tables:
    full_name = f"voyage.dbt_dev.{tbl}"
    print(f"\n{'='*60}")
    print(f"  {full_name}  sample rows")
    print(f"{'='*60}")
    df = spark.sql(f"SELECT * FROM {full_name} LIMIT 5")
    display(df)

In [0]:
tables = [
    "dim_date",
    "dim_tenant",
    "dim_campaign",
    "dim_property",
    "dim_customer",
    "fct_campaign_daily",
    "fct_marketing_funnel_daily",
    "fct_marketing_attribution_daily",
    "fct_website_daily",
    "fct_booking_daily",
    "fct_property_daily",
    "fct_tenant_performance",
]

for tbl in tables:
    full_name = f"voyage.dbt_dev.{tbl}"
    df = spark.sql(f"SELECT * FROM {full_name} LIMIT 1")
    print(f"\n{'='*60}")
    print(f"  {full_name}  — columns ({len(df.columns)})")
    print(f"{'='*60}")
    for col_name, col_type in df.dtypes:
        print(f"  - {col_name}: {col_type}")

## 2. Source Date Range Validation

This notebook validates the date ranges for the three data sources using **voyage.dbt_dev** fact tables:

* **Booking source** — `voyage.dbt_dev.fct_booking_daily` — expected: historical 2014–2017
* **Marketing source** — `voyage.dbt_dev.fct_campaign_daily` — expected: synthetic 2026-04-01 → 2026-09-30
* **Website source** — `voyage.dbt_dev.fct_website_daily` — expected: synthetic 2026-04-01 → 2026-09-30

### 2.1 Booking Source — `fct_booking_daily`

Expected range: historical 2014–2017.

In [0]:
%sql
-- Booking source: fct_booking_daily -- Expected range: 2014–2017
SELECT
    MIN(booking_date)  AS min_booking_date,
    MAX(booking_date)  AS max_booking_date,
    COUNT(*)           AS total_rows,
    COUNT(DISTINCT booking_date) AS distinct_dates
FROM voyage.dbt_dev.fct_booking_daily;

In [0]:
%sql
-- Booking: yearly distribution to confirm 2014–2017 coverage
SELECT
    YEAR(booking_date) AS booking_year,
    COUNT(*)           AS row_count,
    MIN(booking_date)  AS year_min_date,
    MAX(booking_date)  AS year_max_date
FROM voyage.dbt_dev.fct_booking_daily
GROUP BY YEAR(booking_date)
ORDER BY booking_year;

### 2.2 Marketing Source — `fct_campaign_daily`

Expected range: synthetic 2026-04-01 → 2026-09-30.

In [0]:
%sql
-- Marketing source: fct_campaign_daily -- Expected range: 2026-04-01 → 2026-09-30
SELECT
    MIN(date)  AS min_date,
    MAX(date)  AS max_date,
    COUNT(*)   AS total_rows,
    COUNT(DISTINCT date) AS distinct_dates
FROM voyage.dbt_dev.fct_campaign_daily;

In [0]:
%sql
-- Marketing: monthly distribution to confirm 2026-04 through 2026-09 coverage
SELECT
    YEAR(date)  AS yr,
    MONTH(date) AS mo,
    COUNT(*)    AS row_count,
    MIN(date)   AS month_min_date,
    MAX(date)   AS month_max_date
FROM voyage.dbt_dev.fct_campaign_daily
GROUP BY YEAR(date), MONTH(date)
ORDER BY yr, mo;

### 2.3 Website Source — `fct_website_daily`

Expected range: synthetic 2026-04-01 → 2026-09-30.

In [0]:
%sql
-- Website source: fct_website_daily -- Expected range: 2026-04-01 → 2026-09-30
SELECT
    MIN(date)  AS min_date,
    MAX(date)  AS max_date,
    COUNT(*)   AS total_rows,
    COUNT(DISTINCT date) AS distinct_dates
FROM voyage.dbt_dev.fct_website_daily;

In [0]:
%sql
-- Website: monthly distribution to confirm 2026-04 through 2026-09 coverage
SELECT
    YEAR(date)  AS yr,
    MONTH(date) AS mo,
    COUNT(*)    AS row_count,
    MIN(date)   AS month_min_date,
    MAX(date)   AS month_max_date
FROM voyage.dbt_dev.fct_website_daily
GROUP BY YEAR(date), MONTH(date)
ORDER BY yr, mo;

## 3. Summary & Findings

| Source | Table | Expected Range | Actual Min | Actual Max | Status |
| --- | --- | --- | --- | --- | --- |
| Booking | `fct_booking_daily` | 2014-2017 | 2013-06-24 | 2017-08-31 |  See notes |
| Marketing | `fct_campaign_daily` | 2026-04-01 → 2026-09-30 | 2026-04-01 | 2026-09-30 | Exact match |
| Website | `fct_website_daily` | 2026-04-01 → 2026-09-30 | 2026-04-01 | 2026-09-30 | Exact match |

### Booking source notes

* 1 outlier row in **2013** (single date: 2013-06-24) — falls outside the expected 2014-2017 window.
* **2014** starts 2014-03-18 (not Jan 1) and ends 2014-11-18 — partial year.
* **2015** and **2016** have full Jan 1 → Dec 31 coverage.
* **2017** ends 2017-08-31 — partial year (no Sep-Dec data).

### Marketing & Website sources

* Both sources cover exactly **2026-04-01 through 2026-09-30** with all 6 months (Apr–Sep) fully represented.
* 183 distinct dates = 183 calendar days in the range, confirming no gaps.

## 4. dbt Model Generation

The following cells contain the dbt model generation code used to create the staging and marts layer files for the website silver tables. Both cells are commented out for reference — the actual dbt model files live in the dbt project directory.

### 4.1 dbt Staging Model Generation

Generated dbt staging layer files for all three silver fact tables:

* `sources.yml` — source definitions with column tests (not_null, unique, relationships)
* `stg_web_sessions.sql` / `.yml` — staged sessions with campaign FK relationship
* `stg_web_page_views.sql` / `.yml` — staged page views with session FK relationship
* `stg_web_events.sql` / `.yml` — staged events with session FK and event value casting

Each SQL model selected directly from the silver source and was materialized as a view.

In [0]:
'''
# ---------------------------------------------------------
"""
# dbt staging model files for website silver tables
# Copy each block below into the corresponding file.
# ---------------------------------------------------------

print("=" * 70)
print("1. dbt/models/staging/sources.yml  (add to existing Silver source)")
print("=" * 70)
print("""
  - name: fact_web_session
    description: "Deduplicated website sessions from bronze"
    columns:
      - name: session_id
        description: "Unique session identifier"
        tests: [not_null, unique]
      - name: tenant_id
        tests: [not_null]
      - name: session_start
        tests: [not_null]
      - name: session_end
        tests: [not_null]
      - name: source
        tests: [not_null]
      - name: medium
        tests: [not_null]
      - name: campaign_id
        tests:
          - relationships:
              to: ref('stg_campaigns')
              field: campaign_id
              # null values are allowed (organic/direct traffic)

  - name: fact_web_page_view
    description: "Deduplicated website page views from bronze"
    columns:
      - name: page_view_id
        tests: [not_null, unique]
      - name: tenant_id
        tests: [not_null]
      - name: session_id
        tests:
          - not_null
          - relationships:
              to: ref('stg_web_sessions')
              field: session_id
      - name: page_viewed_at
        tests: [not_null]

  - name: fact_web_event
    description: "Deduplicated website events from bronze"
    columns:
      - name: event_id
        tests: [not_null, unique]
      - name: tenant_id
        tests: [not_null]
      - name: session_id
        tests:
          - not_null
          - relationships:
              to: ref('stg_web_sessions')
              field: session_id
      - name: event_timestamp
        tests: [not_null]
      - name: event_name
        tests: [not_null]
""")

print()
print("=" * 70)
print("2. dbt/models/staging/stg_web_sessions.sql")
print("=" * 70)
print("""
{{ config(materialized='view') }}

select
    campaign_id,
    session_id,
    tenant_id,
    user_id,
    device,
    country,
    landing_page,
    medium,
    source,
    session_start,
    session_end,
    updated_at
from {{ source('silver', 'fact_web_session') }}
""")

print()
print("=" * 70)
print("3. dbt/models/staging/stg_web_sessions.yml")
print("=" * 70)
print("""
version: 2

models:
  - name: stg_web_sessions
    description: "Cleaned and renamed website sessions from silver"
    columns:
      - name: session_id
        description: "Unique session identifier"
        tests: [not_null, unique]
      - name: tenant_id
        tests: [not_null]
      - name: user_id
        description: "User identifier"
      - name: device
        description: "Device type (mobile, desktop, tablet)"
      - name: country
        description: "Country code"
      - name: landing_page
        description: "First page viewed in the session"
      - name: medium
        tests: [not_null]
      - name: source
        tests: [not_null]
      - name: campaign_id
        description: "Campaign identifier, nullable for organic/direct traffic"
        tests:
          - relationships:
              to: ref('stg_campaigns')
              field: campaign_id
      - name: session_start
        tests: [not_null]
      - name: session_end
        tests: [not_null]
      - name: updated_at
""")

print()
print("=" * 70)
print("4. dbt/models/staging/stg_web_page_views.sql")
print("=" * 70)
print("""
{{ config(materialized='view') }}

select
    page_view_id,
    session_id,
    tenant_id,
    page_path,
    page_title,
    referrer,
    duration_seconds,
    page_viewed_at,
    updated_at
from {{ source('silver', 'fact_web_page_view') }}
""")

print()
print("=" * 70)
print("5. dbt/models/staging/stg_web_page_views.yml")
print("=" * 70)
print("""
version: 2

models:
  - name: stg_web_page_views
    description: "Cleaned and renamed website page views from silver"
    columns:
      - name: page_view_id
        description: "Unique page view identifier"
        tests: [not_null, unique]
      - name: session_id
        tests:
          - not_null
          - relationships:
              to: ref('stg_web_sessions')
              field: session_id
      - name: tenant_id
        tests: [not_null]
      - name: page_path
        description: "URL path of the viewed page"
      - name: page_title
        description: "Title of the viewed page"
      - name: referrer
        description: "Referrer URL"
      - name: duration_seconds
        description: "Time spent on page in seconds"
      - name: page_viewed_at
        tests: [not_null]
      - name: updated_at
""")

print()
print("=" * 70)
print("6. dbt/models/staging/stg_web_events.sql")
print("=" * 70)
print("""
{{ config(materialized='view') }}

select
    event_id,
    session_id,
    tenant_id,
    event_name,
    cast(event_value as decimal(14,2)) as event_value,
    page_path,
    event_timestamp,
    updated_at
from {{ source('silver', 'fact_web_event') }}
""")

print()
print("=" * 70)
print("7. dbt/models/staging/stg_web_events.yml")
print("=" * 70)
print("""
version: 2

models:
  - name: stg_web_events
    description: "Cleaned and renamed website events from silver"
    columns:
      - name: event_id
        description: "Unique event identifier"
        tests: [not_null, unique]
      - name: session_id
        tests:
          - not_null
          - relationships:
              to: ref('stg_web_sessions')
              field: session_id
      - name: tenant_id
        tests: [not_null]
      - name: event_name
        description: "Event type (search_availability, view_property, begin_booking, purchase)"
        tests: [not_null]
      - name: event_value
        description: "Event payload value, nullable"
      - name: page_path
        description: "URL path where the event was triggered"
      - name: event_timestamp
        tests: [not_null]
      - name: updated_at
""")

print()
print("All dbt staging files generated.")
"""
'''

### 4.2 dbt Marts Model Generation

Generated dbt marts layer files for website analytics:

* `fct_web_session.sql` / `.yml` — session-level fact table with engagement and conversion metrics (page views, events, searches, property views, booking starts, conversions) sourced from `int_web_session_metrics`
* `fct_website_daily.sql` / `.yml` — daily aggregation by tenant, source, medium, and campaign with computed conversion rate, enriched with campaign name/channel/type from `stg_campaigns`

Both marts models were materialized as tables for query performance.

In [0]:
'''
# ---------------------------------------------------------
# dbt marts model files for website analytics
# Copy each block below into the corresponding file.
# ---------------------------------------------------------

print("=" * 70)
print("1. dbt/models/marts/fct_web_session.sql")
print("=" * 70)
print("""
{{ config(materialized='table') }}

select
    session_id,
    tenant_id,
    user_id,
    session_start,
    session_end,
    source,
    medium,
    campaign_id,
    device,
    country,
    landing_page,
    page_view_count,
    event_count,
    search_count,
    property_view_count,
    booking_start_count,
    conversion_count,
    conversion_value,
    engaged_session,
    session_duration_seconds,
    updated_at
from {{ ref('int_web_session_metrics') }}
""")

print()
print("=" * 70)
print("2. dbt/models/marts/fct_web_session.yml")
print("=" * 70)
print("""
version: 2

models:
  - name: fct_web_session
    description: "Session-level fact table with engagement and conversion metrics"
    columns:
      - name: session_id
        description: "Unique session identifier"
        tests: [not_null, unique]
      - name: tenant_id
        tests: [not_null]
      - name: user_id
        description: "User identifier"
      - name: session_start
        tests: [not_null]
      - name: session_end
        tests: [not_null]
      - name: source
        tests: [not_null]
      - name: medium
        tests: [not_null]
      - name: campaign_id
        description: "Campaign identifier, nullable for organic/direct traffic"
      - name: device
        description: "Device type (mobile, desktop, tablet)"
      - name: country
        description: "Country code"
      - name: landing_page
        description: "First page viewed in the session"
      - name: page_view_count
        description: "Number of page views in the session"
      - name: event_count
        description: "Total events fired in the session"
      - name: search_count
        description: "Number of search_availability events"
      - name: property_view_count
        description: "Number of view_property events"
      - name: booking_start_count
        description: "Number of begin_booking events"
      - name: conversion_count
        description: "Number of purchase events"
      - name: conversion_value
        description: "Sum of purchase event values"
      - name: engaged_session
        description: "1 if session had 2+ page views or any events, else 0"
      - name: session_duration_seconds
        description: "Duration from session_start to session_end in seconds"
      - name: updated_at
""")

print()
print("=" * 70)
print("3. dbt/models/marts/fct_website_daily.sql")
print("=" * 70)
print("""
{{ config(materialized='table') }}

with session_metrics as (

    select
        date(session_start) as date,
        tenant_id,
        source,
        medium,
        campaign_id,

        count(*)                             as sessions,
        sum(engaged_session)                 as engaged_sessions,
        sum(page_view_count)                 as page_views,
        sum(event_count)                     as events,
        sum(search_count)                    as searches,
        sum(property_view_count)             as property_views,
        sum(booking_start_count)            as booking_starts,
        sum(conversion_count)                as conversions,
        round(sum(conversion_value), 2)      as conversion_value,

        try_divide(
            sum(conversion_count),
            count(*)
        ) as conversion_rate

    from {{ ref('fct_web_session') }}
    group by
        date(session_start),
        tenant_id,
        source,
        medium,
        campaign_id

),

campaign_lookup as (

    select
        campaign_id,
        campaign_name,
        channel,
        campaign_type
    from {{ ref('stg_campaigns') }}

)

select
    sm.date,
    sm.tenant_id,
    sm.source,
    sm.medium,
    sm.campaign_id,

    sm.sessions,
    sm.engaged_sessions,
    sm.page_views,
    sm.events,
    sm.searches,
    sm.property_views,
    sm.booking_starts,
    sm.conversions,
    sm.conversion_value,
    sm.conversion_rate,

    cl.campaign_name,
    cl.channel,
    cl.campaign_type

from session_metrics sm
left join campaign_lookup cl
    on sm.campaign_id = cl.campaign_id
""")

print()
print("=" * 70)
print("4. dbt/models/marts/fct_website_daily.yml")
print("=" * 70)
print("""
version: 2

models:
  - name: fct_website_daily
    description: "Daily website metrics aggregated by tenant, source, medium, and campaign"
    columns:
      - name: date
        description: "Calendar date derived from session_start"
        tests: [not_null]
      - name: tenant_id
        tests: [not_null]
      - name: source
        tests: [not_null]
      - name: medium
        tests: [not_null]
      - name: campaign_id
        description: "Campaign identifier, nullable for organic/direct traffic"
      - name: sessions
        description: "Total number of sessions"
      - name: engaged_sessions
        description: "Sessions with 2+ page views or at least one event"
      - name: page_views
        description: "Total page views across all sessions"
      - name: events
        description: "Total events fired across all sessions"
      - name: searches
        description: "Total search_availability events"
      - name: property_views
        description: "Total view_property events"
      - name: booking_starts
        description: "Total begin_booking events"
      - name: conversions
        description: "Total purchase events"
      - name: conversion_value
        description: "Sum of purchase event values"
      - name: conversion_rate
        description: "Conversions divided by sessions"
      - name: campaign_name
        description: "Campaign name from stg_campaigns"
      - name: channel
        description: "Campaign channel from stg_campaigns"
      - name: campaign_type
        description: "Campaign type from stg_campaigns"
""")

print()
print("All dbt marts files generated.")
'''

## 5. Website Silver vs. dbt Reconciliation

Built the silver fact tables and generated the dbt marts models in the previous cells. Before calling this pipeline done, ran a reconciliation query to make sure nothing drifted between layers. The query compared five metrics — sessions, page views, events, conversions, and conversion value — between the silver source tables and `voyage.dbt_dev.fct_web_session`. If all five matched, the dbt transformations preserved the data correctly. If any metric was off, something in the staging or marts logic dropped or duplicated rows and needed fixing before this notebook was complete.

### 5.1 Silver vs dbt Mart Reconciliation

In [0]:
%sql
with silver_sessions as (
    select count(*) as sessions
    from voyage.silver.fact_web_session
),

silver_page_views as (
    select count(*) as page_views
    from voyage.silver.fact_web_page_view
),

silver_events as (
    select count(*) as events
    from voyage.silver.fact_web_event
),

silver_conversions as (
    select
        count(*) as conversions,
        round(sum(coalesce(event_value, 0)), 2) as conversion_value
    from voyage.silver.fact_web_event
    where event_name = 'purchase'
),

session_mart as (
    select
        count(*) as sessions,
        sum(page_view_count) as page_views,
        sum(event_count) as events,
        sum(conversion_count) as conversions,
        round(sum(conversion_value), 2) as conversion_value
    from voyage.dbt_dev.fct_web_session
)

select
    case
        when ss.sessions = sm.sessions
         and sp.page_views = sm.page_views
         and se.events = sm.events
         and sc.conversions = sm.conversions
         and sc.conversion_value = sm.conversion_value
        then 'PASS'
        else 'FAIL'
    end as reconciliation_status,

    ss.sessions as silver_sessions,
    sm.sessions as mart_sessions,

    sp.page_views as silver_page_views,
    sm.page_views as mart_page_views,

    se.events as silver_events,
    sm.events as mart_events,

    sc.conversions as silver_conversions,
    sm.conversions as mart_conversions,

    sc.conversion_value as silver_conversion_value,
    sm.conversion_value as mart_conversion_value

from silver_sessions ss
cross join silver_page_views sp
cross join silver_events se
cross join silver_conversions sc
cross join session_mart sm

### 5.2 Marketing Consistency: Orphan Campaign Check

The reconciliation test in the previous cell confirmed row counts matched between silver and the mart. Then checked the campaign dimension side — sessions in `fct_web_session` that carried a `campaign_id` but had no matching record in `stg_campaigns`. These orphan campaigns would break any join to campaign metadata downstream and cause missing campaign names in the daily report. Ran a left join between the mart and `stg_campaigns` on `campaign_id`, filtering for non-null campaign IDs where the campaign table came back null. Expected zero orphans — anything above zero meant a campaign was tracked in the session data but never made it into the campaigns dimension.

In [0]:
%sql
select count(*) as orphan_campaign_sessions
from voyage.dbt_dev.fct_web_session s
left join voyage.dbt_dev.stg_campaigns c
    on s.campaign_id = c.campaign_id
where s.campaign_id is not null
  and c.campaign_id is null

## 6. Marketing Data Mart Validation

Queries and reconciliation checks for the marketing fact table after dbt transformation.

### 6.1 Preview & Detail Queries

Preview the marketing fact table and inspect campaign-level metrics.

In [0]:
%sql
SELECT * FROM voyage.dbt_dev.fct_campaign_daily LIMIT 20;

In [0]:
%sql
select
    campaign_id,
    date,
    impressions,
    clicks,
    spend,
    conversions,
    revenue,
    ctr,
    cpc,
    conversion_rate,
    roas
from voyage.dbt_dev.fct_campaign_daily
order by campaign_id, date;

In [0]:
%sql
select
    campaign_id,
    date,
    revenue,
    ctr,
    cpc,
    conversion_rate,
    roas
from voyage.dbt_dev.fct_campaign_daily
where campaign_id = 'CMP001'
  and date = '2026-04-01';

### 6.2 Aggregate Totals & Reconciliation

Sum all marketing metrics and reconcile silver vs. dbt mart to confirm no data drift.

In [0]:
%sql
select
    sum(impressions) as impressions,
    sum(clicks) as clicks,
    round(sum(spend), 2) as spend,
    sum(conversions) as conversions,
    round(sum(revenue), 2) as revenue
from voyage.dbt_dev.fct_campaign_daily;

In [0]:
%sql
with silver as (   
    select                         
        count(*) as row_count,
        sum(impressions) as impressions,
        sum(clicks) as clicks,
        round(sum(spend), 2) as spend,
        sum(conversions) as conversions,
        round(sum(revenue), 2) as revenue
    from voyage.silver.fact_campaign_daily
),                   
mart as (            
    select              
        count(*) as row_count,
        sum(impressions) as impressions,
        sum(clicks) as clicks,
        round(sum(spend), 2) as spend,
        sum(conversions) as conversions,
        round(sum(revenue), 2) as revenue
    from voyage.dbt_dev.fct_campaign_daily
)
select
    case
        when s.row_count = m.row_count
         and s.impressions = m.impressions
         and s.clicks = m.clicks
         and s.spend = m.spend
         and s.conversions = m.conversions
         and s.revenue = m.revenue
        then 'PASS'
        else 'FAIL'
    end as reconciliation_status,

    s.row_count as silver_rows,
    m.row_count as mart_rows,
    s.impressions - m.impressions as impressions_diff,
    s.clicks - m.clicks as clicks_diff,
    round(s.spend - m.spend, 2) as spend_diff,
    s.conversions - m.conversions as conversions_diff,
    round(s.revenue - m.revenue, 2) as revenue_diff

from silver s
cross join mart m;

### 6.3 Domain Validation

Comprehensive validation of the marketing fact table: row count, grain, metric ranges, and null checks.

In [0]:
%sql
with base as (
    select *
    from voyage.dbt_dev.fct_campaign_daily 
),                            
campaign_check as (                 
    select                
        count(distinct campaign_id) as campaigns,
        count(distinct tenant_id) as tenants,
        count(*) as rows,             
        min(date) as min_date,                                       
        max(date) as max_date       
    from base   
),              
grain_check as (
    select count(*) as duplicate_groups
    from (
        select
            tenant_id,
            campaign_id,
            date
        from base
        group by tenant_id, campaign_id, date
        having count(*) > 1
    )
),
metric_check as (
    select count(*) as invalid_rows
    from base
    where impressions < 0
       or clicks < 0
       or conversions < 0
       or spend < 0
       or revenue < 0
       or clicks > impressions
       or conversions > clicks
       or ctr < 0
       or ctr > 1
       or cpc < 0
       or conversion_rate < 0
       or conversion_rate > 1
       or roas < 0
),
null_check as (
    select count(*) as rows_with_nulls
    from base
    where campaign_id is null
       or tenant_id is null
       or date is null
       or impressions is null
       or clicks is null
       or spend is null
       or conversions is null
       or revenue is null
       or ctr is null
       or cpc is null
       or conversion_rate is null
       or roas is null
)
select
    case
        when c.campaigns = 6
         and c.tenants = 2
         and c.rows = 1098
         and c.min_date = '2026-04-01'
         and c.max_date = '2026-09-30'
         and g.duplicate_groups = 0
         and m.invalid_rows = 0
         and n.rows_with_nulls = 0
        then 'PASS'
        else 'FAIL'
    end as marketing_domain_status,
    c.campaigns,
    c.tenants,
    c.rows,
    c.min_date,
    c.max_date,
    g.duplicate_groups,
    m.invalid_rows,
    n.rows_with_nulls
from campaign_check c
cross join grain_check g
cross join metric_check m
cross join null_check n;

### 6.4 Channel Performance

Aggregate marketing metrics by campaign channel with computed ROAS.

In [0]:
%sql
select
    channel,
    count(distinct campaign_id) as campaigns,
    count(*) as campaign_days,
    sum(impressions) as impressions,
    sum(clicks) as clicks,
    round(sum(spend), 2) as spend,
    sum(conversions) as conversions,
    round(sum(revenue), 2) as revenue,
    round(sum(revenue) / nullif(sum(spend), 0), 2) as aggregate_roas
from voyage.dbt_dev.fct_campaign_daily
group by channel
order by channel;

## 7. Final Pipeline Validation

Final sanity check — verifies that every key dbt_dev fact table has rows. If any table is empty, the pipeline failed and the cell raises a `RuntimeError`.

In [0]:
from pyspark.sql import functions as F


CHECKS = {
    "historical_booking_fact": "voyage.dbt_dev.fct_booking_daily",
    "campaign_daily": "voyage.dbt_dev.fct_campaign_daily",
    "website_daily": "voyage.dbt_dev.fct_website_daily",
    "marketing_funnel": "voyage.dbt_dev.fct_marketing_funnel_daily",
    "marketing_attribution": "voyage.dbt_dev.fct_marketing_attribution_daily",
}


results = []

for name, table_name in CHECKS.items():

    row_count = (
        spark.sql(
            f"SELECT COUNT(*) AS row_count FROM {table_name}"
        )
        .collect()[0]["row_count"]
    )

    results.append(
        (
            name,
            table_name,
            row_count,
        )
    )

    if row_count <= 0:
        raise RuntimeError(
            f"Pipeline validation failed: {table_name} has no rows."
        )


validation_df = spark.createDataFrame(
    results,
    [
        "check_name",
        "table_name",
        "row_count",
    ],
)

display(validation_df)

print(
    "VoyageData orchestration validation passed."
)